In [1]:
import os
from dotenv import load_dotenv, find_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import random
import csv
import json
import time
import numpy as np
import config

In [2]:
load_dotenv(find_dotenv())
engine = create_engine(f'postgresql://{config.db_username}:{config.db_password}@{config.db_host}:{config.db_port}/{config.db_name}')

In [11]:
folders = ['sdss_relational2']
# folders = ['db_2023']

# not_upload = ['field', 'frame', 'mangadapall', 'mangadrpall', 'photoobjall', 'photoz', 'specobjall','galspecextra','galspecindx','zoospec']
# not_upload = ['dbobjects', 'elredshift', 'field', 'frame','galspecindx', 'galspecinfo', 'galspecline','photoobjall', 'photoz', 'photozrf','speclineall', 'speclineindex', 'specobjall', 'spplines']
not_upload = ['field', 'frame','photoz', 'photozrf','galspecextra','galspecindx','platex','samplesizes','specobjall','zoospec']
#upload = ['field', 'frame']

for folder in folders:
    folder_path = f'../data/{folder}/'
    for filename in os.listdir(folder_path):
        csv_file_path = os.path.join(folder_path, filename)
        # if filename != "PhotoObjAll_3.csv":
        #     continue
        if os.path.isfile(csv_file_path):
            pre_table_name = os.path.splitext(filename)[0].lower()
            table_name = pre_table_name.split('_')[0] if ('_' in pre_table_name) else pre_table_name
            full_table_name = '.'.join([folder,table_name])
            if table_name not in not_upload:
            # if table_name in upload:
                df = pd.read_csv(csv_file_path, low_memory=False)
                df.columns = df.columns.str.lower()

                 # Convert boolean columns to '1' or '0'
                for col in df.select_dtypes(include=['bool']).columns:
                    df[col] = df[col].apply(lambda x: '1' if x else '0')

                # Convert uint64 columns to strings or another suitable type
                for col in df.columns:
                    if df[col].dtype == 'uint64':
                        df[col] = df[col].astype('str')  # Convert to str to avoid overflow issues

                df.to_sql(table_name, engine, schema=folder, if_exists='append', index=False)
                print(f"Data inserted successfully into {full_table_name} from {filename}")

Data inserted successfully into sdss_relational2.photoobjall from PhotoObjAll_egallinucci_1.csv


#### Auxiliar code: for data replication queries

In [4]:
schema = 'db_2023'

In [ ]:
with engine.connect() as connection:
    query = text(f"SELECT distinct specobjid FROM {schema}.galspecextra")
    result = connection.execute(query)
    aux_list = [row[0] for row in result.fetchall()]

In [6]:
id_list = ', '.join(f"{str(item)}" for item in aux_list)

In [10]:
query = f"select top 5032 * from galspecindx where specobjid in ({id_list})"
print(query)

select top 5032 * from galspecindx where specobjid in (299492426224003072, 299492700632147968, 299492975510054912, 299492975979816960, 299494075491444736, 299494350369351680, 299494624777496576, 299494899655403520, 299496274514700288, 299496548922845184, 299501772072839168, 299503421340280832, 299504245504239616, 299504520382146560, 299506170119350272, 299506444527495168, 299506994283309056, 299507269630978048, 299507544508884992, 299507818917029888, 299508369142605824, 299508643550750720, 299508644020512768, 299508918428657664, 299509193306564608, 299509193776326656, 299509468184471552, 299510017940285440, 299510018410047488, 299510567696099328, 299510568165861376, 299510843043768320, 299511117921675264, 299511392329820160, 299511667677489152, 299515240620517376, 299515241090279424, 299515515498424320, 299517440113534976, 299517989399586816, 299518264277493760, 299518539155400704, 299518539625162752, 299519089380976640, 299519363789121536, 299519364258883584, 299519638667028480, 29951

In [33]:
file_path = "../Files/queryaux.txt"
with open(file_path, 'w') as file:
    file.write(query)